#### Error Mitigated QSVM (ZNE + REM) - Spambase - Consistent with Ideal/Noisy

This notebook implements two error mitigation techniques:
1. **Zero-Noise Extrapolation (ZNE)**: Extrapolates results from multiple noise scales to estimate zero-noise expectation
2. **Readout Error Mitigation (REM)**: Corrects measurement errors using calibration-based mitigation matrix

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score

In [ ]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load Dataset

In [ ]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", "word_freq_our",
    "word_freq_over", "word_freq_remove", "word_freq_internet", "word_freq_order", "word_freq_mail",
    "word_freq_receive", "word_freq_will", "word_freq_people", "word_freq_report", "word_freq_addresses",
    "word_freq_free", "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money", "word_freq_hp",
    "word_freq_hpl", "word_freq_george", "word_freq_650", "word_freq_lab", "word_freq_labs",
    "word_freq_telnet", "word_freq_857", "word_freq_data", "word_freq_415", "word_freq_85",
    "word_freq_technology", "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project", "word_freq_re",
    "word_freq_edu", "word_freq_table", "word_freq_conference", "char_freq_;", "char_freq_(",
    "char_freq_[", "char_freq_!", "char_freq_$", "char_freq_#", "capital_run_length_average",
    "capital_run_length_longest", "capital_run_length_total", "label"
]

# --- 1. Load the Spambase Dataset (LOCAL PATH) ---
# file_path = '/kaggle/input/spambase/spambase.data'
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

##### Noise Model Factory Functions (ZNE + REM)

In [ ]:
# Base error rates (realistic NISQ device)
NOISE_CONFIGS = {
    'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
    'standard': {'p_1q': 0.001,  'p_2q': 0.01,  'p_ro': 0.02},
    'high': {'p_1q': 0.005,  'p_2q': 0.05,  'p_ro': 0.10}
}

def get_scaled_noise_model(scale_factor=1.0, level='standard', include_readout=True):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    
    Args:
        scale_factor: Noise scaling factor for ZNE (1.0 = base noise)
        level: Noise level ('low', 'standard', 'high')
        include_readout: If True, include readout errors in noise model
    
    Returns:
        noise_model, backend, pass_manager, config
    """
    config = NOISE_CONFIGS.get(level, NOISE_CONFIGS['standard'])
    
    p_1q = config['p_1q']
    p_2q = config['p_2q']
    p_ro = config['p_ro']

    p_1q_scaled = 1 - (1 - p_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_2q)**scale_factor
    p_ro_scaled = 1 - (1 - p_ro)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    
    if include_readout:
        readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
        noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, config

print("Noise model factory function ready!")
print("Available Levels: 'low', 'standard', 'high'")

In [ ]:
# ==========================================
# READOUT ERROR MITIGATION (REM) FUNCTIONS
# ==========================================

def build_rem_matrix(p_ro, n_qubits):
    """
    Build the readout error mitigation (inverse calibration) matrix.
    
    For symmetric readout error where:
    - P(measure 1 | prepared 0) = p_ro
    - P(measure 0 | prepared 1) = p_ro
    
    The single-qubit calibration matrix A is:
    A = [[1-p_ro, p_ro],
         [p_ro, 1-p_ro]]
    
    For n qubits (assuming independent errors), the full matrix is:
    A_full = A ⊗ A ⊗ ... ⊗ A (n times)
    
    REM applies A_full^(-1) to correct the noisy probability distribution.
    
    Args:
        p_ro: Readout error probability
        n_qubits: Number of qubits
    
    Returns:
        rem_matrix: Inverse calibration matrix for REM
    """
    # Single qubit calibration matrix
    A_1q = np.array([[1 - p_ro, p_ro],
                     [p_ro, 1 - p_ro]])
    
    # Build full calibration matrix via tensor product
    A_full = A_1q
    for _ in range(n_qubits - 1):
        A_full = np.kron(A_full, A_1q)
    
    # Compute inverse (mitigation matrix)
    try:
        rem_matrix = np.linalg.inv(A_full)
    except np.linalg.LinAlgError:
        # Use pseudo-inverse if singular
        rem_matrix = np.linalg.pinv(A_full)
    
    return rem_matrix


def apply_rem_to_kernel(kernel_matrix, p_ro, n_qubits):
    """
    Apply Readout Error Mitigation to a kernel matrix.
    
    Since the quantum kernel is estimated from measurement outcomes,
    REM improves the fidelity estimation by correcting measurement biases.
    
    For the fidelity-based kernel, REM effectively rescales the kernel values
    to compensate for the measurement error-induced bias.
    
    The correction formula for fidelity with symmetric readout error:
    K_corrected = (K_noisy - bias) / (1 - 2*p_ro)
    
    where bias accounts for the error contribution to the overlap.
    
    Args:
        kernel_matrix: Noisy kernel matrix from quantum computation
        p_ro: Readout error probability
        n_qubits: Number of qubits used in the feature map
    
    Returns:
        corrected_kernel: REM-corrected kernel matrix
    """
    # For fidelity-based kernels with symmetric readout error,
    # the correction factor is derived from the confusion matrix
    # Each qubit contributes a factor of (1 - 2*p_ro) to the true fidelity
    
    # Effective correction factor for n-qubit measurement
    correction_factor = (1 - 2 * p_ro) ** n_qubits
    
    if abs(correction_factor) < 1e-10:
        print("Warning: Correction factor too small, using raw kernel")
        return kernel_matrix
    
    # The noisy fidelity F_noisy relates to true fidelity F_true by:
    # F_noisy = correction_factor * F_true + (1 - correction_factor) * 0.5
    # (for symmetric noise, random measurement gives ~0.5 overlap on average)
    
    # Solving for F_true:
    # F_true = (F_noisy - 0.5 * (1 - correction_factor)) / correction_factor
    
    bias = 0.5 * (1 - correction_factor)
    corrected_kernel = (kernel_matrix - bias) / correction_factor
    
    # Clip to valid kernel range [0, 1] for numerical stability
    corrected_kernel = np.clip(corrected_kernel, 0, 1)
    
    # Ensure diagonal is exactly 1 (self-similarity)
    if kernel_matrix.shape[0] == kernel_matrix.shape[1]:
        np.fill_diagonal(corrected_kernel, 1.0)
    
    return corrected_kernel


print("REM (Readout Error Mitigation) functions ready!")
print("Available functions: build_rem_matrix(), apply_rem_to_kernel()")

##### Experiment Configurations (ZNE + REM)

In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (ZNE + REM ABLATION STUDY)
# ==========================================
# 
# Mitigation Types:
#   - 'ZNE': Zero-Noise Extrapolation only
#   - 'REM': Readout Error Mitigation only
#   - 'ZNE+REM': Both techniques combined
#   - 'NONE': No mitigation (baseline noisy)
#
# ==========================================

experiments = [
    # =====================================================
    # SECTION A: ZNE EXPERIMENTS (Original)
    # =====================================================
    
    # --- EXP 1: Sample Size Effect (ZNE) ---
    {'id': 'ZNE_100samp',  'samples': 100, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_300samp',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_500samp',  'samples': 500, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},

    # --- EXP 2: Dimensionality Effect (ZNE) ---
    {'id': 'ZNE_2feat',   'samples': 300, 'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_4feat',   'samples': 300, 'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_6feat',   'samples': 300, 'k_features': 6,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_8feat',   'samples': 300, 'k_features': 8,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_10feat',  'samples': 300, 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_12feat',  'samples': 300, 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},

    # --- EXP 3: Shot Noise Effect (ZNE) ---
    {'id': 'ZNE_128shots',  'samples': 300, 'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_512shots',  'samples': 300, 'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_1024shots', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},

    # --- EXP 4: Reps Effect (ZNE) ---
    {'id': 'ZNE_Reps1', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_Reps2', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_Reps3', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 3, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},

    # --- EXP 5: Entanglement Ablation (ZNE) ---
    {'id': 'ZNE_Linear',   'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear',   'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_Circular', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_Full',     'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full',     'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},

    # --- EXP 6: Noise Level Ablation (ZNE) ---
    {'id': 'ZNE_LowNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low',      'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_StdNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_HighNoise', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high',     'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},

    # --- EXP 7: ZNE Scale Ablation ---
    {'id': 'ZNE_LinearExtrap',    'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNE_QuadraticExtrap', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE', 'zne_scales': [1.0, 2.0, 3.0]},
    {'id': 'ZNE_NoExtrap',        'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'NONE', 'zne_scales': [1.0]},

    # =====================================================
    # SECTION B: REM EXPERIMENTS (New)
    # =====================================================
    
    # --- EXP 8: Sample Size Effect (REM) ---
    {'id': 'REM_100samp',  'samples': 100, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_300samp',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_500samp',  'samples': 500, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},

    # --- EXP 9: Dimensionality Effect (REM) ---
    {'id': 'REM_2feat',   'samples': 300, 'k_features': 2,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_4feat',   'samples': 300, 'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_6feat',   'samples': 300, 'k_features': 6,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_8feat',   'samples': 300, 'k_features': 8,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_10feat',  'samples': 300, 'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_12feat',  'samples': 300, 'k_features': 12, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},

    # --- EXP 10: Shot Noise Effect (REM) ---
    {'id': 'REM_128shots',  'samples': 300, 'k_features': 8, 'shots': 128,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_512shots',  'samples': 300, 'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_1024shots', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},

    # --- EXP 11: Noise Level Ablation (REM) ---
    {'id': 'REM_LowNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low',      'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_StdNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'REM', 'zne_scales': [1.0]},
    {'id': 'REM_HighNoise', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high',     'mitigation': 'REM', 'zne_scales': [1.0]},

    # =====================================================
    # SECTION C: COMBINED ZNE+REM EXPERIMENTS
    # =====================================================
    
    # --- EXP 12: Sample Size Effect (ZNE+REM) ---
    {'id': 'ZNEREM_100samp',  'samples': 100, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNEREM_200samp',  'samples': 200, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNEREM_300samp',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNEREM_500samp',  'samples': 500, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},

    # --- EXP 13: Noise Level Ablation (ZNE+REM) ---
    {'id': 'ZNEREM_LowNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low',      'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNEREM_StdNoise',  'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard', 'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},
    {'id': 'ZNEREM_HighNoise', 'samples': 300, 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high',     'mitigation': 'ZNE+REM', 'zne_scales': [1.0, 3.0]},
]

# Count experiments by type
zne_count = sum(1 for e in experiments if e['mitigation'] == 'ZNE')
rem_count = sum(1 for e in experiments if e['mitigation'] == 'REM')
combined_count = sum(1 for e in experiments if e['mitigation'] == 'ZNE+REM')
none_count = sum(1 for e in experiments if e['mitigation'] == 'NONE')

print(f"Total experiments configured: {len(experiments)}")
print(f"  - ZNE only: {zne_count}")
print(f"  - REM only: {rem_count}")
print(f"  - ZNE+REM combined: {combined_count}")
print(f"  - No mitigation (baseline): {none_count}")

##### Main Experiment Loop (with ZNE + REM Support)

In [ ]:
from qiskit_machine_learning.utils import algorithm_globals

all_results = []

for i, config in enumerate(experiments, 1):
    mitigation_type = config.get('mitigation', 'ZNE')
    
    print("="*80)
    print(f"EXPERIMENT {i}/{len(experiments)}: {config['id']} ({mitigation_type} Mitigated)")
    print("="*80)
    print(f"  Samples: {config['samples']}")
    print(f"  K Features: {config['k_features']}")
    print(f"  Noise Level: {config['noise_level']}")
    print(f"  Mitigation: {mitigation_type}")
    if 'ZNE' in mitigation_type:
        print(f"  ZNE Scales: {config['zne_scales']}")
    
    start_time = time.time()
    
    # --- 1. Data Preparation ---
    X = df.drop('label', axis=1)
    y = df['label']
    
    X_subset, _, y_subset, _ = train_test_split(
        X, y,
        train_size=config['samples'],
        stratify=y,
        random_state=42
    )
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        test_size=0.30,
        random_state=42,
        stratify=y_subset
    )
    
    # Scaling & Feature Selection
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)
    
    # Drop Correlated
    corr_matrix = X_train_scaled.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
    X_train_scaled.drop(columns=to_drop, inplace=True)
    X_test_scaled.drop(columns=to_drop, inplace=True)
    
    # SelectKBest
    k = config['k_features']
    selector = SelectKBest(score_func=f_classif, k=k)
    X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
    X_test_kbest = selector.transform(X_test_scaled)
    
    # Get noise config for REM
    noise_config = NOISE_CONFIGS.get(config['noise_level'], NOISE_CONFIGS['standard'])
    p_ro = noise_config['p_ro']
    n_qubits = k  # Number of qubits = number of features
    
    # --- 2. Compute Kernels ---
    kernels_train = {}
    kernels_test = {}
    
    feature_map = ZZFeatureMap(
        feature_dimension=k, 
        reps=config['reps'], 
        entanglement=config['entanglement']
    )
    
    for scale in config['zne_scales']:
        print(f"  Computing kernel for scale={scale}...")
        _, backend, pm, _ = get_scaled_noise_model(
            scale_factor=scale, 
            level=config['noise_level'],
            include_readout=True  # Always include readout noise in simulation
        )
        sampler = AerSampler.from_backend(backend=backend, default_shots=config['shots'])
        fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
        qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
        
        kernels_train[scale] = qkernel.evaluate(x_vec=X_train_kbest)
        kernels_test[scale] = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
    
    # --- 3. Apply Mitigation ---
    scales = config['zne_scales']
    
    if mitigation_type == 'NONE':
        # No mitigation - use raw noisy kernel
        kernel_train_final = kernels_train[scales[0]]
        kernel_test_final = kernels_test[scales[0]]
        print("  No mitigation applied (baseline noisy).")
        
    elif mitigation_type == 'ZNE':
        # ZNE only - Richardson Extrapolation
        if len(scales) == 1:
            kernel_train_final = kernels_train[scales[0]]
            kernel_test_final = kernels_test[scales[0]]
        elif scales == [1.0, 3.0]:
            # Linear 2-point extrapolation
            kernel_train_final = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
            kernel_test_final = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
        elif scales == [1.0, 2.0, 3.0]:
            # Quadratic 3-point extrapolation
            kernel_train_final = 3.0 * kernels_train[1.0] - 3.0 * kernels_train[2.0] + 1.0 * kernels_train[3.0]
            kernel_test_final = 3.0 * kernels_test[1.0] - 3.0 * kernels_test[2.0] + 1.0 * kernels_test[3.0]
        else:
            raise ValueError(f"Unsupported ZNE scales: {scales}")
        print("  ZNE mitigation applied.")
        
    elif mitigation_type == 'REM':
        # REM only - Apply to raw kernel
        kernel_train_final = apply_rem_to_kernel(kernels_train[scales[0]], p_ro, n_qubits)
        kernel_test_final = apply_rem_to_kernel(kernels_test[scales[0]], p_ro, n_qubits)
        print(f"  REM mitigation applied (p_ro={p_ro}, n_qubits={n_qubits}).")
        
    elif mitigation_type == 'ZNE+REM':
        # Combined: First apply ZNE, then REM
        # Step 1: ZNE extrapolation
        if scales == [1.0, 3.0]:
            kernel_train_zne = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
            kernel_test_zne = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
        elif scales == [1.0, 2.0, 3.0]:
            kernel_train_zne = 3.0 * kernels_train[1.0] - 3.0 * kernels_train[2.0] + 1.0 * kernels_train[3.0]
            kernel_test_zne = 3.0 * kernels_test[1.0] - 3.0 * kernels_test[2.0] + 1.0 * kernels_test[3.0]
        else:
            kernel_train_zne = kernels_train[scales[0]]
            kernel_test_zne = kernels_test[scales[0]]
        
        # Step 2: REM correction on ZNE result
        kernel_train_final = apply_rem_to_kernel(kernel_train_zne, p_ro, n_qubits)
        kernel_test_final = apply_rem_to_kernel(kernel_test_zne, p_ro, n_qubits)
        print(f"  ZNE+REM combined mitigation applied.")
    
    else:
        raise ValueError(f"Unknown mitigation type: {mitigation_type}")
    
    # Ensure kernel values are valid
    kernel_train_final = np.clip(kernel_train_final, 0, 1)
    kernel_test_final = np.clip(kernel_test_final, 0, 1)
    
    # --- 4. Train SVC ---
    param_grid = {'C': [0.1, 1, 10, 100]}
    svc = SVC(kernel='precomputed', class_weight='balanced')
    grid = GridSearchCV(svc, param_grid, cv=3, scoring='accuracy')
    grid.fit(kernel_train_final, y_train)
    best_model = grid.best_estimator_
    
    # --- 5. Evaluation ---
    y_train_pred = best_model.predict(kernel_train_final)
    y_test_pred = best_model.predict(kernel_test_final)
    
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    elapsed_time = time.time() - start_time
    
    print(f"  → Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    print(f"  → Spam Recall: {recall:.4f} | Gen Gap: {gen_gap:.4f}")
    print(f"  → Time: {elapsed_time:.1f}s")
    
    all_results.append({
        'id': config['id'],
        'samples': config['samples'],
        'k_features': config['k_features'],
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'noise_level': config['noise_level'],
        'mitigation': mitigation_type,
        'zne_scales': str(config['zne_scales']),
        'train_acc': train_acc,
        'test_acc': test_acc,
        'spam_recall': recall,
        'gen_gap': gen_gap,
        'best_c': grid.best_params_['C'],
        'time_seconds': elapsed_time
    })

# Save Results
results_df = pd.DataFrame(all_results)
results_df.to_csv('em_qsvm_results.csv', index=False)
print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE!")
print("Results saved to: em_qsvm_results.csv")
print("="*80)

##### Results Summary and Visualization

In [ ]:
# Display summary table
print("\nResults Summary:")
print(results_df[['id', 'mitigation', 'samples', 'k_features', 'noise_level', 'test_acc', 'spam_recall', 'gen_gap']].to_string(index=False))

In [ ]:
# ==========================================
# VISUALIZATION: Compare Mitigation Methods
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Filter to comparable experiments (300 samples, 8 features, standard noise)
baseline_filter = (results_df['samples'] == 300) & (results_df['k_features'] == 8) & (results_df['noise_level'] == 'standard')
baseline_df = results_df[baseline_filter].copy()

if len(baseline_df) > 0:
    # Group by mitigation type
    mitigation_summary = baseline_df.groupby('mitigation').agg({
        'test_acc': 'mean',
        'spam_recall': 'mean',
        'gen_gap': 'mean'
    }).reset_index()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot 1: Test Accuracy by Mitigation
    colors = {'NONE': '#d62728', 'ZNE': '#1f77b4', 'REM': '#ff7f0e', 'ZNE+REM': '#2ca02c'}
    bar_colors = [colors.get(m, '#333333') for m in mitigation_summary['mitigation']]
    
    axes[0].bar(mitigation_summary['mitigation'], mitigation_summary['test_acc'], color=bar_colors)
    axes[0].set_ylabel('Test Accuracy')
    axes[0].set_title('Test Accuracy by Mitigation Method')
    axes[0].set_ylim(0, 1)
    
    # Plot 2: Spam Recall by Mitigation
    axes[1].bar(mitigation_summary['mitigation'], mitigation_summary['spam_recall'], color=bar_colors)
    axes[1].set_ylabel('Spam Recall')
    axes[1].set_title('Spam Recall by Mitigation Method')
    axes[1].set_ylim(0, 1)
    
    # Plot 3: Generalization Gap by Mitigation
    axes[2].bar(mitigation_summary['mitigation'], mitigation_summary['gen_gap'], color=bar_colors)
    axes[2].set_ylabel('Generalization Gap')
    axes[2].set_title('Generalization Gap (Lower is Better)')
    
    plt.tight_layout()
    plt.savefig('mitigation_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nMitigation Method Summary:")
    print(mitigation_summary.to_string(index=False))
else:
    print("No baseline experiments found for visualization.")

In [ ]:
# ==========================================
# HEATMAP: All Experiments Overview
# ==========================================

plt.figure(figsize=(14, 12))

# Prepare heatmap data
heatmap_data = results_df.set_index('id')[['test_acc', 'spam_recall', 'gen_gap', 'train_acc']]

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            center=0.75, linewidths=.5, cbar_kws={'label': 'Score'})

plt.title('Error Mitigation Experiments Summary (ZNE + REM)', fontsize=14, pad=20)
plt.ylabel('Experiment ID')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('em_experiments_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()